# 📋 Day 3 Cheatsheet — Python Internals, Visualization & Real Apps

> Quick-reference notebook. Every pattern you need, fully annotated.
> Run each cell independently to verify your understanding.

| Section | Topics |
|---------|--------|
| 1 | Tricky Python Concepts |
| 2 | Decorators |
| 3 | Standard Library (datetime / collections / itertools / pathlib / json) |
| 4 | Matplotlib |
| 5 | Streamlit Patterns |
| 6 | FastAPI + re + os + logging |

---
# 🔵 1 — Tricky Python Concepts

In [ ]:
# ── 1.1 MUTABLE DEFAULT ARGUMENTS ──────────────────────────────────────────
# Rule: NEVER use a mutable (list/dict/set) as a default argument.
#       Use None instead.

def bad(item, cart=[]):          # ❌ cart is shared across ALL calls
    cart.append(item)
    return cart

def good(item, cart=None):       # ✅ fresh list per call
    if cart is None:
        cart = []
    cart.append(item)
    return cart

print(bad('a'))    # ['a']
print(bad('b'))    # ['a', 'b']  ← bug!
print(good('a'))   # ['a']
print(good('b'))   # ['b']       ← correct

In [ ]:
# ── 1.2 is vs == ────────────────────────────────────────────────────────────
# ==    → same VALUE
# is    → same OBJECT in memory (identity)
# Rule: Only use 'is' to check for None (or True/False in rare cases)

a = [1, 2, 3]; b = [1, 2, 3]
print(a == b)         # True  — same value
print(a is b)         # False — different objects

x = None
print(x is None)      # ✅ correct
print(x == None)      # ⚠️  works but not Pythonic

In [ ]:
# ── 1.3 *args and **kwargs ──────────────────────────────────────────────────
# *args   → tuple of extra positional args
# **kwargs → dict of extra keyword args

def demo(required, *args, **kwargs):
    print(f"required={required}, args={args}, kwargs={kwargs}")

demo('hello', 1, 2, 3, color='red', size='L')
# required=hello, args=(1, 2, 3), kwargs={'color': 'red', 'size': 'L'}

# Unpacking when CALLING a function
nums = [1, 2, 3]
opts = {'color': 'blue', 'size': 'M'}
demo('hi', *nums, **opts)            # unpacks list → positional, dict → keyword

In [ ]:
# ── 1.4 List Comprehension vs Generator ─────────────────────────────────────
import sys

lst = [x**2 for x in range(100_000)]    # builds entire list in memory
gen = (x**2 for x in range(100_000))    # lazy, one value at a time

print(f"list: {sys.getsizeof(lst):,} bytes")
print(f"gen:  {sys.getsizeof(gen):,} bytes")   # tiny!

# Use generator directly with sum/max/min/any/all
total = sum(x**2 for x in range(100_000))   # no [] needed

# Generator function with yield
def count_up(start, stop):
    while start < stop:
        yield start              # pauses here, resumes on next()
        start += 1

for n in count_up(1, 5):
    print(n, end=' ')   # 1 2 3 4

In [ ]:
# ── 1.5 Shallow vs Deep Copy ────────────────────────────────────────────────
import copy

original = [[1, 2], [3, 4]]

alias   = original              # ← same object, not a copy at all
shallow = copy.copy(original)   # ← new outer list, shared inner lists
deep    = copy.deepcopy(original)  # ← fully independent copy

original[0][0] = 99

print(f"alias:   {alias}")    # [[99,2],[3,4]] ← changed (same object)
print(f"shallow: {shallow}")  # [[99,2],[3,4]] ← changed (shared inner)
print(f"deep:    {deep}")     # [[1,2],[3,4]]  ← unchanged ✅

# Rule: nested structures → always deepcopy

In [ ]:
# ── 1.6 global and nonlocal ─────────────────────────────────────────────────
# LEGB: Local → Enclosing → Global → Built-in

counter = 0

def inc_global():
    global counter     # modify module-level variable
    counter += 1

inc_global(); inc_global()
print(f"global counter: {counter}")   # 2

# nonlocal — for closures (nested function modifying enclosing scope)
def make_counter():
    count = 0
    def inc():
        nonlocal count     # reach into enclosing function
        count += 1
        return count
    return inc

c = make_counter()
print(c(), c(), c())   # 1 2 3

In [ ]:
# ── 1.7 Walrus Operator := (Python 3.8+) ────────────────────────────────────
# Assigns AND returns a value in one expression

import re

data = "Error: disk full"

# Without walrus
m = re.search(r'Error: (.+)', data)
if m: print(m.group(1))

# With walrus — cleaner
if m := re.search(r'Error: (.+)', data):
    print(m.group(1))

# While loop — classic use case
import io
f = io.StringIO("a\nb\nc")
while line := f.readline():
    print(line.strip(), end=' ')  # a b c

# In comprehensions — compute once, filter and use
data = [1, -4, 6, -9, 3]
result = [y for x in data if (y := abs(x)) > 3]
print(f"\nabs > 3: {result}")  # [4, 6, 9]

---
# 🟡 2 — Decorators

In [ ]:
# ── 2.1 Basic Decorator Pattern ─────────────────────────────────────────────
# @decorator is syntactic sugar for: func = decorator(func)

from functools import wraps

def my_decorator(func):      # 1. takes a function
    @wraps(func)             # 2. preserve __name__, __doc__
    def wrapper(*args, **kwargs):    # 3. wrapper accepts any signature
        print(f"Before {func.__name__}")
        result = func(*args, **kwargs)
        print(f"After  {func.__name__}")
        return result        # 4. always return the original result
    return wrapper           # 5. return the wrapper function

@my_decorator
def greet(name):
    return f"Hello {name}"

print(greet("Alice"))
print(greet.__name__)   # 'greet' ← preserved by @wraps

In [ ]:
# ── 2.2 Practical Decorators ────────────────────────────────────────────────
from functools import wraps
import time

# Timer
def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[timer] {func.__name__}: {time.perf_counter()-t0:.4f}s")
        return result
    return wrapper

# Logger
def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[log] {func.__name__}({args}, {kwargs})")
        return func(*args, **kwargs)
    return wrapper

@timer
@log_call        # decorators apply bottom-up: log_call first, then timer
def compute(n):
    return sum(range(n))

print(compute(1_000_000))

In [ ]:
# ── 2.3 Decorator with Arguments ────────────────────────────────────────────
# Pattern: add one more outer layer that accepts the config

from functools import wraps

def repeat(times=2):             # 1. factory — takes config
    def decorator(func):         # 2. actual decorator
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator             # 3. return the decorator

@repeat(times=3)
def say(msg):
    print(f"  {msg}")

say("hello")

# ─────

# Retry decorator
import time

def retry(times=3, delay=0.5):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times+1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  Attempt {attempt} failed: {e}")
                    if attempt < times: time.sleep(delay)
            raise RuntimeError(f"{func.__name__} failed after {times} attempts")
        return wrapper
    return decorator

In [ ]:
# ── 2.4 @property ───────────────────────────────────────────────────────────
# Turns a method into a computed attribute with optional getter/setter/deleter

class Circle:
    def __init__(self, radius):
        self._radius = radius

    @property
    def radius(self):              # getter
        return self._radius

    @radius.setter
    def radius(self, value):       # setter with validation
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value

    @property
    def area(self):                # computed, read-only property
        import math
        return math.pi * self._radius ** 2

c = Circle(5)
print(f"radius={c.radius}, area={c.area:.2f}")
c.radius = 10                      # calls setter
print(f"radius={c.radius}, area={c.area:.2f}")
try:
    c.radius = -1                  # triggers ValueError
except ValueError as e:
    print(f"Error: {e}")

---
# 🟢 3 — Standard Library

In [ ]:
# ── 3.1 datetime ────────────────────────────────────────────────────────────
from datetime import datetime, date, timedelta

now   = datetime.now()
today = date.today()

# Format: strftime — string FROM time
print(now.strftime('%Y-%m-%d'))           # '2024-01-15'
print(now.strftime('%d/%m/%Y %H:%M:%S')) # '15/01/2024 14:30:00'
print(now.strftime('%B %d, %Y'))          # 'January 15, 2024'

# Parse: strptime — string PARSE time
dt = datetime.strptime('2024-03-15', '%Y-%m-%d')
print(dt)   # 2024-03-15 00:00:00

# Arithmetic
tomorrow    = today + timedelta(days=1)
next_month  = today + timedelta(weeks=4)
diff        = date(2024, 12, 31) - date(2024, 1, 1)
print(f"Days in year: {diff.days}")

In [ ]:
# ── 3.2 collections ─────────────────────────────────────────────────────────
from collections import Counter, defaultdict, namedtuple, deque

# Counter — frequency count
c = Counter(['a','b','a','c','a','b'])
print(c)                           # Counter({'a':3,'b':2,'c':1})
print(c.most_common(2))            # [('a',3),('b',2)]
print(c['z'])                      # 0 — no KeyError for missing keys

# defaultdict — auto-create missing keys
d = defaultdict(list)              # missing key → []
for word in ['apple','avocado','banana']:
    d[word[0]].append(word)
print(dict(d))   # {'a': ['apple','avocado'], 'b': ['banana']}

# namedtuple — lightweight immutable record
Point = namedtuple('Point', ['x', 'y'])
p = Point(3, 4)
print(p.x, p.y)   # access by name

# deque — O(1) append/pop from both ends
dq = deque([1,2,3], maxlen=3)     # maxlen auto-removes oldest
dq.append(4)                       # [2,3,4]
dq.appendleft(0)                   # [0,2,3]
print(dq)

In [ ]:
# ── 3.3 itertools ───────────────────────────────────────────────────────────
import itertools

# chain — flatten multiple iterables
print(list(itertools.chain([1,2],[3,4],[5,6])))           # [1,2,3,4,5,6]
print(list(itertools.chain.from_iterable([[1,2],[3,4]]))) # [1,2,3,4]

# islice — lazy slice without materializing
print(list(itertools.islice(range(1_000_000), 5)))   # [0,1,2,3,4]

# product — Cartesian product (replaces nested loops)
print(list(itertools.product(['red','blue'],['S','M'])))  # all combos

# combinations / permutations
items = ['A','B','C']
print(list(itertools.combinations(items, 2)))   # order doesn't matter
print(list(itertools.permutations(items, 2)))   # order matters

# groupby — group consecutive equal elements (sort FIRST)
data = sorted([('a',1),('b',2),('a',3)], key=lambda x: x[0])
for k, g in itertools.groupby(data, key=lambda x: x[0]):
    print(k, list(g))

In [ ]:
# ── 3.4 pathlib ─────────────────────────────────────────────────────────────
from pathlib import Path

p = Path('/home/alice/data/sales.csv')

# Properties
print(p.name)     # 'sales.csv'
print(p.stem)     # 'sales'
print(p.suffix)   # '.csv'
print(p.parent)   # /home/alice/data

# Build paths with / — cross-platform!
output = Path.cwd() / 'reports' / 'Q4.csv'

# File I/O
tmp = Path('/tmp/test.txt')
tmp.write_text("hello")         # write string
print(tmp.read_text())          # read string
tmp.unlink()                    # delete

# Checks
print(p.exists())               # False (file doesn't exist)
print(p.is_file(), p.is_dir())

# Create directory tree
Path('/tmp/a/b/c').mkdir(parents=True, exist_ok=True)

# Glob — find files by pattern
py_files = list(Path('.').glob('**/*.py'))   # recursive
print(f"Found {len(py_files)} .py files")

In [ ]:
# ── 3.5 json ────────────────────────────────────────────────────────────────
import json
from pathlib import Path

data = {'name': 'Alice', 'scores': [95, 87], 'active': True, 'address': None}

# Encode: Python → JSON string
s = json.dumps(data, indent=2)   # pretty print
print(s)

# Decode: JSON string → Python
obj = json.loads(s)
print(type(obj).__name__)   # dict

# File I/O
p = Path('/tmp/data.json')
with open(p, 'w') as f: json.dump(data, f, indent=2)   # write
with open(p) as f:       loaded = json.load(f)           # read
p.unlink()

# Custom encoder for non-JSON types (e.g. date, datetime, numpy)
from datetime import date

class SmartEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, date): return obj.isoformat()
        return super().default(obj)

print(json.dumps({'date': date(2024,3,15)}, cls=SmartEncoder))

---
# 🟠 4 — Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

# Shared data
months = ['Jan','Feb','Mar','Apr','May','Jun']
sales  = [12, 19, 15, 22, 28, 35]

print("Setup done.")

In [ ]:
# ── 4.1 All 5 Chart Types + Formatting Template ──────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 1. Line — trends over time
axes[0,0].plot(months, sales, marker='o', color='steelblue', linewidth=2)
axes[0,0].set_title('Line — plt.plot()')

# 2. Bar — category comparison
axes[0,1].bar(months, sales, color='steelblue', alpha=0.8, edgecolor='white')
axes[0,1].set_title('Bar — plt.bar()')

# 3. Scatter — correlation
np.random.seed(0)
x = np.random.randn(50); y = x * 1.5 + np.random.randn(50)
axes[0,2].scatter(x, y, alpha=0.6, c='purple', s=40)
axes[0,2].set_title('Scatter — plt.scatter()')

# 4. Histogram — distribution
data = np.random.normal(50, 15, 500)
axes[1,0].hist(data, bins=25, color='green', edgecolor='white', alpha=0.8)
axes[1,0].axvline(data.mean(), color='red', linestyle='--', label=f'Mean={data.mean():.1f}')
axes[1,0].legend()
axes[1,0].set_title('Histogram — plt.hist()')

# 5. Multiple lines on one plot
returns = [2,3,1,4,5,6]
axes[1,1].plot(months, sales,   label='Sales',   color='steelblue')
axes[1,1].plot(months, returns, label='Returns', color='tomato', linestyle='--')
axes[1,1].legend()
axes[1,1].set_title('Multi-line')

# 6. Formatting reference
axes[1,2].set_title('Formatting Cheatsheet', fontsize=11)
text = (
    "ax.set_title('Title', fontsize=14)\n"
    "ax.set_xlabel('X label')\n"
    "ax.set_ylabel('Y label')\n"
    "ax.set_ylim(0, 100)\n"
    "ax.legend()\n"
    "plt.tight_layout()\n"
    "fig.savefig('out.png', dpi=150)"
)
axes[1,2].text(0.05, 0.5, text, transform=axes[1,2].transAxes,
               fontfamily='monospace', fontsize=9, verticalalignment='center')
axes[1,2].axis('off')

fig.suptitle('Matplotlib Quick Reference', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.2 Saving Figures ──────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(months, sales, marker='o')
ax.set_title('Chart to Save')

plt.tight_layout()

fig.savefig('/tmp/chart.png',  dpi=150, bbox_inches='tight')  # PNG — web/slides
fig.savefig('/tmp/chart.pdf',  bbox_inches='tight')           # PDF — print
fig.savefig('/tmp/chart.svg',  bbox_inches='tight')           # SVG — vector

print("Saved PNG, PDF, SVG")
plt.show()

# Key parameters:
# dpi=150   → medium quality; dpi=300 for print
# bbox_inches='tight' → don't clip labels/titles

---
# 🔴 5 — Streamlit Patterns

> These cells show code — run them as `.py` files with `streamlit run filename.py`

In [ ]:
# ── 5. COMPLETE STREAMLIT CHEATSHEET ────────────────────────────────────────
# Save as streamlit_ref.py and run: streamlit run streamlit_ref.py

STREAMLIT_REF = '''
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

# ── PAGE CONFIG (must be first st call) ──
st.set_page_config(page_title="App", layout="wide", page_icon="📊")

# ── TEXT ──
st.title("Page Title")
st.header("Section"); st.subheader("Sub")
st.write("auto-detect type: text, df, dict, fig...")
st.markdown("**bold** | *italic* | `code`")
st.code("print('hello')", language='python')
st.caption("small grey text"); st.divider()

# ── ALERTS ──
st.success("Done!"); st.warning("Watch out"); st.error("Failed"); st.info("FYI")

# ── DATA DISPLAY ──
df = pd.DataFrame({'A': [1,2,3], 'B': [4,5,6]})
st.dataframe(df)                  # interactive, sortable
st.table(df)                      # static
st.json({'key': 'value'})         # formatted JSON

# ── METRICS ──
col1, col2, col3 = st.columns(3)
col1.metric("Revenue",  "$45K", "+12%")
col2.metric("Orders",   "1,234", "-3%")
col3.metric("NPS",      "72",   "+5pts")

# ── BUILT-IN CHARTS ──
st.line_chart(df)                 # instant, interactive
st.bar_chart(df)
st.area_chart(df)

# ── MATPLOTLIB ──
fig, ax = plt.subplots()
ax.plot([1,2,3], [4,5,6])
st.pyplot(fig)
plt.close(fig)                    # always close after st.pyplot()

# ── WIDGETS ──
text   = st.text_input("Name", value="Alice")
num    = st.number_input("Count", min_value=0, value=10)
slider = st.slider("Range", 0, 100, (20, 80))   # returns tuple
sel    = st.selectbox("Pick", ["A","B","C"])
multi  = st.multiselect("Multi", ["X","Y","Z"], default=["X"])
check  = st.checkbox("Enable")
radio  = st.radio("Mode", ["Fast","Slow"])
date   = st.date_input("Date")
btn    = st.button("Run")         # True when clicked

# ── FILE UPLOAD ──
uploaded = st.file_uploader("Upload CSV", type=["csv"])
if uploaded:
    df2 = pd.read_csv(uploaded)
    st.dataframe(df2)

# ── DOWNLOAD BUTTON ──
csv = df.to_csv(index=False)
st.download_button("Download CSV", csv, "data.csv", "text/csv")

# ── LAYOUT ──
with st.sidebar:
    st.header("Filters")
    # sidebar widgets here

left, right = st.columns(2)      # equal width columns
left.write("Left"); right.write("Right")

with st.expander("Show details"):
    st.write("Hidden content")

# ── CACHING ──
@st.cache_data                   # cache return value (data/df)
def load_data(path):
    return pd.read_csv(path)

@st.cache_resource               # cache resource (model/connection)
def load_model():
    pass  # return loaded model

# ── SESSION STATE — persist state across reruns ──
if 'count' not in st.session_state:
    st.session_state.count = 0

if st.button("Increment"):
    st.session_state.count += 1

st.write(f"Count: {st.session_state.count}")
'''
print(STREAMLIT_REF)

---
# 🎁 6 — FastAPI + re + os + logging

In [ ]:
# ── 6.1 FastAPI Complete Cheatsheet ─────────────────────────────────────────
# pip install fastapi uvicorn
# uvicorn api:app --reload   (then visit http://localhost:8000/docs)

FASTAPI_REF = '''
from fastapi import FastAPI, HTTPException, Query, Path, Depends
from pydantic import BaseModel, Field
from typing import Optional, List

app = FastAPI(title="My API", version="1.0", description="Docs at /docs")

# ── Pydantic model (request/response validation) ──
class Item(BaseModel):
    name:     str = Field(..., min_length=1, max_length=100)
    price:    float = Field(..., gt=0)
    category: str
    active:   bool = True

# In-memory store (use a real DB in production)
items = {}

# ── GET — read data ──
@app.get("/")
def root():
    return {"message": "API running"}

@app.get("/items")                           # GET /items?category=food&limit=10
def list_items(
    category: Optional[str] = None,
    limit: int = Query(default=10, ge=1, le=100)   # validated query param
):
    result = list(items.values())
    if category:
        result = [i for i in result if i['category'] == category]
    return result[:limit]

@app.get("/items/{item_id}")                 # path parameter
def get_item(item_id: int = Path(..., ge=1)):
    if item_id not in items:
        raise HTTPException(status_code=404, detail="Item not found")
    return items[item_id]

# ── POST — create data ──
@app.post("/items", status_code=201)
def create_item(item: Item):
    item_id = len(items) + 1
    items[item_id] = {"id": item_id, **item.dict()}
    return items[item_id]

# ── PUT — update data ──
@app.put("/items/{item_id}")
def update_item(item_id: int, item: Item):
    if item_id not in items:
        raise HTTPException(status_code=404, detail="Item not found")
    items[item_id].update(item.dict())
    return items[item_id]

# ── DELETE ──
@app.delete("/items/{item_id}", status_code=204)
def delete_item(item_id: int):
    if item_id not in items:
        raise HTTPException(status_code=404, detail="Item not found")
    del items[item_id]

# ── Startup/Shutdown events ──
@app.on_event("startup")
async def startup():
    print("API starting up — connect to DB here")
'''
print(FASTAPI_REF)

In [ ]:
# ── 6.2 re — Regular Expressions Cheatsheet ─────────────────────────────────
import re

text = "Email: alice@example.com | Phone: +91-9876543210 | Date: 2024-03-15"

# Core functions
m   = re.search(r'\d{4}-\d{2}-\d{2}', text)    # first match (or None)
all_ = re.findall(r'\w+@\w+\.\w+', text)         # list of all matches
new  = re.sub(r'\d', '#', text)                  # replace matches
parts= re.split(r'\s*\|\s*', text)               # split by pattern

print(f"search:  {m.group() if m else None}")
print(f"findall: {all_}")
print(f"split:   {parts}")

# Groups
m = re.search(r'(\d{4})-(\d{2})-(\d{2})', text)
print(f"groups:       {m.groups()}")
print(f"group(1,2,3): {m.group(1)}, {m.group(2)}, {m.group(3)}")

# Named groups
m = re.search(r'(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})', text)
print(f"named:  year={m.group('year')}, month={m.group('month')}")

# Compile for repeated use (performance)
EMAIL_RE = re.compile(r'[\w.+-]+@[\w-]+\.[\w.]+')
print(f"email:  {EMAIL_RE.findall(text)}")

print()
# ── Common Patterns Reference ──
patterns = {
    'email':   r'[\w.+-]+@[\w-]+\.[\w.]+',
    'phone':   r'[+\d][\d\s-]{9,}',
    'date':    r'\d{4}-\d{2}-\d{2}',
    'url':     r'https?://[\w./-]+',
    'ip':      r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
    'number':  r'-?\d+(\.\d+)?',
    'word':    r'\b\w+\b',
}
for name, pat in patterns.items():
    print(f"  {name:<8} {pat}")

In [ ]:
# ── 6.3 os — OS Interface Cheatsheet ────────────────────────────────────────
import os

# Environment variables — the right way to handle secrets
os.environ['DB_URL'] = 'postgresql://localhost/mydb'   # set
db_url = os.environ.get('DB_URL', 'sqlite:///default.db')  # get with default
print(f"DB_URL: {db_url}")

# Current state
print(f"cwd:       {os.getcwd()}")
print(f"username:  {os.environ.get('USER', 'unknown')}")

# Path operations (prefer pathlib, but know os.path)
p = '/home/alice/data/sales.csv'
print(f"basename:  {os.path.basename(p)}")
print(f"dirname:   {os.path.dirname(p)}")
print(f"splitext:  {os.path.splitext(p)}")   # ('/home/alice/data/sales', '.csv')
print(f"join:      {os.path.join('/home','alice','data')}")  # cross-platform!

# Directory operations
os.makedirs('/tmp/os_test/sub', exist_ok=True)   # mkdir -p
print(f"exists: {os.path.exists('/tmp/os_test')}")
print(f"isdir:  {os.path.isdir('/tmp/os_test')}")
os.listdir('/tmp')[:5]   # list directory

# Run system command (use subprocess for complex cases)
exit_code = os.system('echo "hello from os.system"')
print(f"exit code: {exit_code}")

In [ ]:
# ── 6.4 logging — Production Logging Cheatsheet ─────────────────────────────
import logging

# ── Quick setup ──
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('myapp')

# 5 levels (low → high severity)
logger.debug('debug info — dev only')     # 10
logger.info('normal operation')           # 20
logger.warning('something unexpected')    # 30
logger.error('something failed')          # 40
logger.critical('system is broken')       # 50

print("---")

# ── Production pattern: file + console ──
def setup_logger(name, log_file='/tmp/app.log', console_level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)          # capture everything

    fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(name)s | %(message)s')

    ch = logging.StreamHandler()            # console
    ch.setLevel(console_level)             # INFO+ to console
    ch.setFormatter(fmt)

    fh = logging.FileHandler(log_file)     # file
    fh.setLevel(logging.DEBUG)             # DEBUG+ to file
    fh.setFormatter(fmt)

    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger

prod = setup_logger('pipeline')
prod.info('Pipeline started — 10,000 rows to process')
prod.debug('Config: batch_size=100, workers=4')  # only in file
prod.warning('Null values in column: price (12 rows)')

# Log exceptions with traceback
try:
    x = 1 / 0
except ZeroDivisionError:
    prod.exception('Unexpected error during processing')  # includes traceback in log

---
# ⚡ Ultra-Quick Reference Card

## Block 1 — Tricky Concepts
```python
# Mutable default → use None
def f(x, lst=None): lst = lst or []; lst.append(x); return lst

# is vs == → use is ONLY for None
x is None          # ✅ correct None check

# *args / **kwargs
def f(*args, **kwargs): ...       # collect
f(*[1,2,3], **{'a':1})           # unpack

# Generator (lazy, memory-efficient)
gen = (x**2 for x in range(1_000_000))  # () not []
total = sum(x**2 for x in range(1_000_000))  # no brackets needed

# Deep copy for nested structures
import copy; deep = copy.deepcopy(original)

# Walrus operator
if m := re.search(pattern, text): print(m.group())
while chunk := file.read(8192): process(chunk)
```

## Block 2 — Decorators
```python
from functools import wraps

def decorator(func):          # basic
    @wraps(func)
    def wrapper(*args, **kwargs): return func(*args, **kwargs)
    return wrapper

def with_args(config):        # decorator with arguments
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs): return func(*args, **kwargs)
        return wrapper
    return decorator

@property          # getter
@prop.setter       # setter with validation
```

## Block 3 — stdlib
```python
# datetime
datetime.now().strftime('%Y-%m-%d')   # format
datetime.strptime('2024-01-15', '%Y-%m-%d')  # parse
today + timedelta(days=7)             # arithmetic

# collections
Counter(['a','b','a']).most_common(2) # frequency + top-N
defaultdict(list)                     # no KeyError, auto-default
namedtuple('Point', ['x','y'])        # lightweight record
deque(maxlen=N)                       # O(1) ends, fixed size

# itertools
chain.from_iterable([[1,2],[3,4]])    # flatten
islice(huge_iter, 5)                  # lazy slice
product(['A','B'],['X','Y'])          # cartesian product
groupby(sorted_data, key=fn)          # group (sort first!)

# pathlib
Path('/tmp') / 'data' / 'file.csv'   # cross-platform join
p.read_text(), p.write_text(s)        # file I/O
p.mkdir(parents=True, exist_ok=True)  # mkdir -p
list(p.glob('**/*.csv'))              # recursive glob

# json
json.dumps(obj, indent=2)    # → string
json.loads(s)                # ← string
json.dump(obj, f)            # → file
json.load(f)                 # ← file
```

## Block 4 — Matplotlib
```python
fig, ax = plt.subplots(figsize=(10,4))  # always use fig, ax
ax.plot(x, y, marker='o', color='steelblue', linewidth=2, label='Sales')
ax.bar(cats, vals, color='steelblue', alpha=0.8)
ax.scatter(x, y, c='purple', alpha=0.7, s=60)
ax.hist(data, bins=25, color='green', edgecolor='white')
ax.set_title('Title'); ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.legend(); ax.set_ylim(0); plt.tight_layout()
fig.savefig('out.png', dpi=150, bbox_inches='tight')

fig, axes = plt.subplots(2, 2, figsize=(12,8))  # subplots
axes[0,0].plot(...)  # access by [row, col]
```

## Block 5 — Streamlit
```python
st.title()  st.header()  st.write()  st.markdown()
st.dataframe(df)  st.table(df)  st.metric('Label', value, delta)
st.line_chart(df)  st.bar_chart(df)  st.pyplot(fig)
st.text_input()  st.slider()  st.selectbox()  st.multiselect()  st.checkbox()
st.button()  st.file_uploader()  st.download_button()
col1, col2 = st.columns(2)
with st.sidebar: ...
with st.expander('Details'): ...
@st.cache_data def load(): ...     # cache data
st.session_state.key = value       # persist state
```

## Bonus — FastAPI + re + os + logging
```python
# FastAPI
@app.get('/items/{id}')  def get(id: int): ...
@app.post('/items')      def create(item: MyModel): ...
raise HTTPException(status_code=404, detail='Not found')

# re
re.search(r'pattern', text)  # first match
re.findall(r'pattern', text) # all matches list
re.sub(r'old', 'new', text)  # replace

# os
os.environ.get('KEY', 'default')     # env vars
os.makedirs('path', exist_ok=True)   # mkdir -p

# logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)
logger.info('msg')  logger.warning('msg')  logger.exception('msg')
```

---
*Day 3 Cheatsheet | Python Internals, Visualization & Real Apps*